# Experiment 2: robustness and causally validated mathematical-error tracing

This notebook runs the complete follow-up specified in `results/experiment2.md`. It reads the frozen Experiment 1 activation cache, writes all new outputs under `artifacts/experiment2`, and never modifies Experiment 1. Use an A100 runtime.

## 1. Authenticate and set up the repository

Add a fine-grained GitHub token with **Contents: read** permission to Colab Secrets as `GITHUB_TOKEN`, and enable notebook access to that secret. The token is passed through a temporary askpass script; it is never embedded in the clone URL, Git remote, notebook source, or cell output. If the repository is already present, the cell updates it only by a fast-forward pull.

In [ ]:
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path


def load_github_token() -> str:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        token = getpass("GitHub token (hidden): ").strip()
    if not token:
        raise RuntimeError(
            "A GitHub token is required. Add GITHUB_TOKEN to Colab Secrets "
            "and enable notebook access, then rerun this cell."
        )
    return token


GITHUB_TOKEN = load_github_token()
ASKPASS_PATH = Path(tempfile.gettempdir()) / "math_error_github_askpass.sh"
ASKPASS_PATH.write_text(
    (
        "#!/bin/sh\n"
        'case "$1" in\n'
        "*Username*) echo x-access-token ;;\n"
        '*Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    ),
    encoding="utf-8",
)
ASKPASS_PATH.chmod(0o700)


def github_git_env() -> dict[str, str]:
    environment = os.environ.copy()
    environment.update({
        "GITHUB_TOKEN": GITHUB_TOKEN,
        "GIT_ASKPASS": str(ASKPASS_PATH),
        "GIT_ASKPASS_REQUIRE": "force",
        "GIT_TERMINAL_PROMPT": "0",
    })
    return environment


REPOSITORY_URL = "https://github.com/sagnikc395/tracing-mathematical-error-detection-in-language-models.git"
AUTHENTICATED_REPOSITORY_URL = REPOSITORY_URL.replace("https://", "https://x-access-token@", 1)
REPOSITORY = Path("/content/tracing-mathematical-error-detection-in-language-models")
if (REPOSITORY / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "pull", "--ff-only"],
        check=True, env=github_git_env(),
    )
elif REPOSITORY.exists():
    raise RuntimeError(
        f"{REPOSITORY} exists but is not a Git checkout. "
        "Delete or rename that stale directory, then rerun this cell."
    )
else:
    subprocess.run(
        ["git", "clone", AUTHENTICATED_REPOSITORY_URL, str(REPOSITORY)],
        check=True, env=github_git_env(),
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPOSITORY)], check=True)
os.chdir(REPOSITORY)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
print(f"Repository: {REPOSITORY}")

## 2. Mount Drive and freeze runtime paths

Experiment 2 requires the full Drive-backed Experiment 1 directory, including `activation_shards`. The compact GitHub artifact package is insufficient. New resumable shards and results are stored in `MyDrive/math-error-tracing/artifacts/experiment2`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/math-error-tracing")
DATA_PATH = DRIVE_ROOT / "data/processbench.jsonl"
EXPERIMENT1_DIR = DRIVE_ROOT / "artifacts/qwen2.5-math-1.5b-a100-bf16"
OUTPUT_DIR = DRIVE_ROOT / "artifacts/experiment2"
CONFIG_PATH = REPOSITORY / "configs/experiment2.yaml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"data": str(DATA_PATH), "experiment1": str(EXPERIMENT1_DIR), "output": str(OUTPUT_DIR)})

In [ ]:
import json
from datetime import datetime, timezone
from time import monotonic

STATUS_PATH = OUTPUT_DIR / "notebook_status.json"
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

def write_status(stage: str, status: str, **extra) -> None:
    payload = {
        "stage": stage,
        "status": status,
        "updated_at": datetime.now(timezone.utc).isoformat(),
        **extra,
    }
    temporary = STATUS_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2))
    temporary.replace(STATUS_PATH)

def run_stage(stage: str) -> None:
    command = [
        sys.executable, "-m", "causal_circuits.experiment2_cli",
        "--config", str(CONFIG_PATH),
        "--experiment1-dir", str(EXPERIMENT1_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--data-path", str(DATA_PATH),
        stage,
    ]
    started = monotonic()
    log_path = LOG_DIR / f"{stage}.colab.log"
    write_status(stage, "running", log_path=str(log_path))
    print(f"\n=== {stage} ===", flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        process = subprocess.Popen(
            command, cwd=REPOSITORY, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        return_code = process.wait()
    elapsed = monotonic() - started
    status = "complete" if return_code == 0 else "failed"
    write_status(stage, status, elapsed_seconds=elapsed, log_path=str(log_path))
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## 3. Validate inputs

This fails immediately if the data, Experiment 1 directions, or complete activation shards are missing.

In [ ]:
run_stage("validate-config")

## 4. Stage A: stronger marker-boundary robustness

This runs the error-only and first-step-centered probes, onset-jump bootstrap, combined surface/metadata control, paired control comparison, leave-one-generator-out tests, PCA, calibration, trajectories, and subgroups. It reuses Experiment 1 activations and requires no model forward passes.

In [ ]:
run_stage("analyze-robustness")
print((OUTPUT_DIR / "marker_robustness/decision.json").read_text())

## 5. Stage B: natural step-end replication

The extraction stores the state at the final non-whitespace token of each written step, before the artificial marker is causally visible. It is sharded and resumable. Within each shard, similar-length traces are batched together to reduce padding, and requested boundary states are transferred to CPU once per batch. These are scheduling optimizations only: all traces, steps, and 29 hidden-state indices are retained. The fitting stage independently selects probe settings on validation data and performs a paired comparison at frozen Experiment 1 index 23.

In [ ]:
run_stage("extract-semantic")
run_stage("fit-semantic")
print((OUTPUT_DIR / "semantic_boundary/decision.json").read_text())

## 6. Stage C: counterbalanced verdict audit

The model is asked for one token under both `A=valid, B=invalid` and the reversed mapping. Canonical invalid-minus-valid margins are averaged, cancelling a stable letter preference. Scoring projects only the final non-padding decoder state instead of materializing unused vocabulary logits at every prompt position. Every completed inference batch is atomically checkpointed; inspect `verdict_audit/progress.json` and `verdict_audit/individual.checkpoint.csv` while it runs. The decision file records whether the validation assay is competent before the causal result is interpreted.

In [ ]:
run_stage("audit-verdict")
print((OUTPUT_DIR / "verdict_audit/decision.json").read_text())

## 7. Stage D: gradient alignment and positive-control interventions

For the same boundary and intervention norm, this compares the frozen learned probe direction against each example's local verdict-gradient direction. The latter is the positive control establishing that the hook and outcome can detect a causal perturbation. The eight non-zero direction/dose variants for each trace/mapping are evaluated in one A100 batch; zero-dose rows reuse the gradient pass baseline. No sample, direction, dose, layer, or mapping is dropped. Each completed trace/mapping job is atomically checkpointed; inspect `causal_validation/progress.json` and the two `*.checkpoint.csv` files while it runs. If a smaller GPU runs out of memory, lower only `causal.batch_size`; existing checkpoints remain valid.

In [ ]:
run_stage("causal-validation")
run_stage("plot")
print((OUTPUT_DIR / "causal_validation/decision.json").read_text())

## 8. Inspect and stage compact artifacts

The Drive directory remains the complete result. This cell copies only tables, decisions, configuration, and figures into the repository's `artifacts/experiment2`; large semantic activation shards are deliberately excluded. It does not commit or push.

In [ ]:
import shutil

import pandas as pd
from IPython.display import display

display(pd.read_csv(OUTPUT_DIR / "marker_robustness/error_only_metrics.csv"))
display(pd.read_csv(OUTPUT_DIR / "semantic_boundary/semantic_vs_marker_paired.csv"))
display(pd.read_csv(OUTPUT_DIR / "verdict_audit/summary.csv"))
display(pd.read_csv(OUTPUT_DIR / "causal_validation/intervention_summary.csv"))

REPOSITORY_ARTIFACTS = REPOSITORY / "artifacts/experiment2"
REPOSITORY_ARTIFACTS.mkdir(parents=True, exist_ok=True)
for source in OUTPUT_DIR.rglob("*"):
    if not source.is_file() or "semantic_activation_shards" in source.parts:
        continue
    relative = source.relative_to(OUTPUT_DIR)
    destination = REPOSITORY_ARTIFACTS / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
print(f"Compact artifacts staged at {REPOSITORY_ARTIFACTS}")